In [1]:
import pandas as pd
hotel = pd.read_csv("hotel.csv")

C:\Users\Dian Pratiwi\AppData\Local\Temp\ipykernel_17272\434615713.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
import nltk #natural language toolkit
from nltk.stem import PorterStemmer
nltk.download("stopwords")
# Initialize Python porter stemmer
ps = PorterStemmer()

[nltk_data] Downloading package stopwords to C:\Users\Dian
[nltk_data]     Pratiwi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# 1. Data cleaning
import string
import re
def clean_data(hotel):
    # Convert text to lowercase
    hotel = hotel.lower() #mengubah tiap kalimat menjadi lowercase ex: GOOD jadi good

    # Remove punctuation
    hotel = hotel.translate(str.maketrans("", "", string.punctuation)) #menghilangkan tanda baca

    # Remove special characters
    hotel = re.sub(r"[^\w\s]", "", hotel)

    # Remove extra whitespace
    hotel = re.sub(r"\s+", " ", hotel) #menghilangkan spasi berlebih

    return hotel

hotel["Review"] = hotel["Review"].apply(clean_data)



In [4]:
# 2. Standardization
from nltk.corpus import stopwords

def standardize_text(hotel):
    # Convert text to lowercase
    hotel = hotel.lower()

    # Remove stop words
    stop_words = set(stopwords.words("english")) #stopwords : an a the
    hotel = " ".join([word for word in hotel.split() if word not in stop_words]) #selain kata stopwords ga dihilangkan

    return hotel

hotel["Review"] = hotel["Review"].apply(standardize_text)



In [5]:
# 3. Stemming
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
def stem_hotel(hotel):
    # Stem words
    hotel = " ".join([stemmer.stem(word) for word in hotel.split()]) #stemming : running jadi run

    return hotel

hotel["Review"] = hotel["Review"].apply(stem_hotel)

print(hotel['Review'])

0        nice hotel expens park got good deal stay hote...
1        ok noth special charg diamond member hilton de...
2        nice room 4 experi hotel monaco seattl good ho...
3        uniqu great stay wonder time hotel monaco loca...
4        great stay great stay went seahawk game awesom...
                               ...                        
20486    best kept secret 3rd time stay charm 5star ca ...
20487    great locat price view hotel great quick place...
20488    ok look nice modern outsid desk staff nt parti...
20489    hotel theft ruin vacat hotel open sept 17 2007...
20490    peopl talk ca nt believ excel rate hotel nt ye...
Name: Review, Length: 20491, dtype: object


In [6]:
import re
import string


def label_sentiment(Rating):
  """Assigns sentiment score based on rating threshold (customizable)."""
  rating_int = int(Rating)
  sentiment_threshold = 3  # Adjust this threshold as needed
  return 1 if rating_int >= sentiment_threshold else 0

  #lebih dari sama dengan 3 maka dilabelling 1
  #kurang dari 3 maka dilabelling 0

# Assuming you have a DataFrame 'data' with a 'Rating' column

# Clean the "Review" column using the clean_data function
hotel["Review"] = hotel["Review"].apply(clean_data)

# Label sentiment in a new "Sentiment" column using label_sentiment
hotel["Sentiment"] = hotel["Rating"].apply(label_sentiment)

# Print the processed data
print(hotel)

Sentiment=hotel["Sentiment"]


                                                  Review  Rating  Sentiment
0      nice hotel expens park got good deal stay hote...       4          1
1      ok noth special charg diamond member hilton de...       2          0
2      nice room 4 experi hotel monaco seattl good ho...       3          1
3      uniqu great stay wonder time hotel monaco loca...       5          1
4      great stay great stay went seahawk game awesom...       5          1
...                                                  ...     ...        ...
20486  best kept secret 3rd time stay charm 5star ca ...       5          1
20487  great locat price view hotel great quick place...       4          1
20488  ok look nice modern outsid desk staff nt parti...       2          0
20489  hotel theft ruin vacat hotel open sept 17 2007...       1          0
20490  peopl talk ca nt believ excel rate hotel nt ye...       2          0

[20491 rows x 3 columns]


In [7]:
#Menghitung skor tf idf
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a TfidfVectorizer object
vectorizer = TfidfVectorizer(max_df=0.8, min_df=2, stop_words='english')

# Transform the data into a TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(hotel['Review'])

# Get the TF-IDF scores for each word
tfidf_scores = tfidf_matrix.toarray()

# Convert the sparse TF-IDF matrix to a dense NumPy array
dense_tfidf_matrix = tfidf_matrix.toarray()


# Print the TF-IDF scores
print(tfidf_scores)


[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]





SKEMA 1 (TS = 0.1936)

In [8]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.1936, random_state=0)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.87      0.63      0.73       620
           1       0.94      0.98      0.96      3348

    accuracy                           0.93      3968
   macro avg       0.90      0.81      0.84      3968
weighted avg       0.92      0.93      0.92      3968



In [9]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.1936, random_state=10)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.86      0.64      0.73       610
           1       0.94      0.98      0.96      3358

    accuracy                           0.93      3968
   macro avg       0.90      0.81      0.85      3968
weighted avg       0.93      0.93      0.92      3968



In [10]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.1936, random_state=42)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.88      0.60      0.72       602
           1       0.93      0.98      0.96      3366

    accuracy                           0.93      3968
   macro avg       0.90      0.79      0.84      3968
weighted avg       0.92      0.93      0.92      3968



In [11]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.1936, random_state=0)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.83      0.74      0.78       620
           1       0.95      0.97      0.96      3348

    accuracy                           0.94      3968
   macro avg       0.89      0.85      0.87      3968
weighted avg       0.93      0.94      0.93      3968



In [12]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.1936, random_state=10)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.81      0.72      0.76       610
           1       0.95      0.97      0.96      3358

    accuracy                           0.93      3968
   macro avg       0.88      0.84      0.86      3968
weighted avg       0.93      0.93      0.93      3968



In [13]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.1936, random_state=42)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.83      0.68      0.75       602
           1       0.94      0.97      0.96      3366

    accuracy                           0.93      3968
   macro avg       0.89      0.83      0.85      3968
weighted avg       0.93      0.93      0.93      3968



Skema 2 (TS = 0.2)


In [15]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.2, random_state=0)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.87      0.63      0.73       640
           1       0.94      0.98      0.96      3459

    accuracy                           0.93      4099
   macro avg       0.90      0.81      0.85      4099
weighted avg       0.92      0.93      0.92      4099



In [16]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.2, random_state=10)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.86      0.63      0.73       629
           1       0.94      0.98      0.96      3470

    accuracy                           0.93      4099
   macro avg       0.90      0.81      0.84      4099
weighted avg       0.92      0.93      0.92      4099



In [17]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.2, random_state=42)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.87      0.61      0.72       625
           1       0.93      0.98      0.96      3474

    accuracy                           0.93      4099
   macro avg       0.90      0.80      0.84      4099
weighted avg       0.92      0.93      0.92      4099



In [18]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.2, random_state=0)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.82      0.74      0.78       640
           1       0.95      0.97      0.96      3459

    accuracy                           0.93      4099
   macro avg       0.89      0.85      0.87      4099
weighted avg       0.93      0.93      0.93      4099



In [19]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.2, random_state=10)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.81      0.71      0.76       629
           1       0.95      0.97      0.96      3470

    accuracy                           0.93      4099
   macro avg       0.88      0.84      0.86      4099
weighted avg       0.93      0.93      0.93      4099



In [20]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.2, random_state=42)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.83      0.69      0.76       625
           1       0.95      0.98      0.96      3474

    accuracy                           0.93      4099
   macro avg       0.89      0.83      0.86      4099
weighted avg       0.93      0.93      0.93      4099



SKEMA 3 (TS = 0.25)


In [22]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.25, random_state=0)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model= LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.86      0.63      0.73       782
           1       0.94      0.98      0.96      4341

    accuracy                           0.93      5123
   macro avg       0.90      0.81      0.84      5123
weighted avg       0.93      0.93      0.92      5123



In [23]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.25, random_state=10)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model= LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.86      0.63      0.72       802
           1       0.93      0.98      0.96      4321

    accuracy                           0.93      5123
   macro avg       0.90      0.80      0.84      5123
weighted avg       0.92      0.93      0.92      5123



In [ ]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.25, random_state=42)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model= LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.88      0.62      0.73       795
           1       0.93      0.99      0.96      4328

    accuracy                           0.93      5123
   macro avg       0.91      0.80      0.84      5123
weighted avg       0.93      0.93      0.92      5123



In [ ]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.25, random_state=0)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.81      0.71      0.76       782
           1       0.95      0.97      0.96      4341

    accuracy                           0.93      5123
   macro avg       0.88      0.84      0.86      5123
weighted avg       0.93      0.93      0.93      5123



In [ ]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.25, random_state=10)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.87      0.63      0.73       802
           1       0.93      0.98      0.96      4321

    accuracy                           0.93      5123
   macro avg       0.90      0.80      0.84      5123
weighted avg       0.92      0.93      0.92      5123



In [ ]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.25, random_state=42)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.84      0.66      0.74       629
           1       0.94      0.98      0.96      3470

    accuracy                           0.93      4099
   macro avg       0.89      0.82      0.85      4099
weighted avg       0.93      0.93      0.93      4099



SKEMA 4 (TS : 0.3)


In [ ]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.3, random_state=0)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.86      0.67      0.75       640
           1       0.94      0.98      0.96      3459

    accuracy                           0.93      4099
   macro avg       0.90      0.83      0.86      4099
weighted avg       0.93      0.93      0.93      4099



In [ ]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.3, random_state=10)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.81      0.70      0.75       961
           1       0.95      0.97      0.96      5187

    accuracy                           0.93      6148
   macro avg       0.88      0.84      0.85      6148
weighted avg       0.92      0.93      0.93      6148



In [ ]:
#SVM LINEAR
#pemodelan svm
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, Sentiment, test_size=0.3, random_state=42)

# Create an SVM model
model = SVC(kernel='linear')

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
print("pemodelan SVM".center(75,'='))

print('CLASSIFICATION REPORT SVM'.center(75,'='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.83      0.68      0.75       955
           1       0.94      0.97      0.96      5193

    accuracy                           0.93      6148
   macro avg       0.89      0.83      0.85      6148
weighted avg       0.93      0.93      0.93      6148



In [ ]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.3, random_state=0)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model= LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.87      0.61      0.72       941
           1       0.93      0.98      0.96      5207

    accuracy                           0.93      6148
   macro avg       0.90      0.80      0.84      6148
weighted avg       0.92      0.93      0.92      6148



In [ ]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.3, random_state=10)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model= LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.87      0.62      0.72       961
           1       0.93      0.98      0.96      5187

    accuracy                           0.93      6148
   macro avg       0.90      0.80      0.84      6148
weighted avg       0.92      0.93      0.92      6148



In [ ]:
#pemodelan regresi logistik
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.3, random_state=42)

print("PEMODELAN DENGAN REGRESI LOGISTIK".center(75,"="))
model= LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('CLASSIFICATION REPORT REGRESI LOGISTIK'.center(75,'='))
print(classification_report(y_test, y_pred))

=====================PEMODELAN DENGAN REGRESI LOGISTIK=====================
===================CLASSIFICATION REPORT REGRESI LOGISTIK==================
              precision    recall  f1-score   support

           0       0.88      0.61      0.72       955
           1       0.93      0.98      0.96      5193

    accuracy                           0.93      6148
   macro avg       0.91      0.80      0.84      6148
weighted avg       0.92      0.93      0.92      6148



NAIVE BAYES

In [28]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.1936, random_state=0)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


MemoryError: Unable to allocate 3.49 GiB for an array with shape (20491, 22863) and data type float64

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.1936, random_state=10)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


========================PEMODELAN DENGAN NAIVE BAYES=======================
=====================CLASSIFICATION REPORT NAIVE BAYES=====================
              precision    recall  f1-score   support

           0       0.18      0.34      0.24       610
           1       0.86      0.72      0.78      3358

    accuracy                           0.66      3968
   macro avg       0.52      0.53      0.51      3968
weighted avg       0.75      0.66      0.70      3968



In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.1936, random_state=42)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


========================PEMODELAN DENGAN NAIVE BAYES=======================
=====================CLASSIFICATION REPORT NAIVE BAYES=====================
              precision    recall  f1-score   support

           0       0.18      0.36      0.24       602
           1       0.86      0.71      0.78      3366

    accuracy                           0.66      3968
   macro avg       0.52      0.54      0.51      3968
weighted avg       0.76      0.66      0.70      3968



In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.2, random_state=0)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


========================PEMODELAN DENGAN NAIVE BAYES=======================
=====================CLASSIFICATION REPORT NAIVE BAYES=====================
              precision    recall  f1-score   support

           0       0.19      0.35      0.25       640
           1       0.86      0.73      0.79      3459

    accuracy                           0.67      4099
   macro avg       0.53      0.54      0.52      4099
weighted avg       0.75      0.67      0.70      4099



In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.2, random_state=10)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


========================PEMODELAN DENGAN NAIVE BAYES=======================
=====================CLASSIFICATION REPORT NAIVE BAYES=====================
              precision    recall  f1-score   support

           0       0.18      0.35      0.24       629
           1       0.86      0.72      0.78      3470

    accuracy                           0.66      4099
   macro avg       0.52      0.53      0.51      4099
weighted avg       0.75      0.66      0.70      4099



In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.2, random_state=42)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


========================PEMODELAN DENGAN NAIVE BAYES=======================
=====================CLASSIFICATION REPORT NAIVE BAYES=====================
              precision    recall  f1-score   support

           0       0.18      0.36      0.24       625
           1       0.86      0.71      0.78      3474

    accuracy                           0.66      4099
   macro avg       0.52      0.54      0.51      4099
weighted avg       0.76      0.66      0.70      4099



In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.25, random_state=0)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.83      0.67      0.74       955
           1       0.94      0.97      0.96      5193

    accuracy                           0.93      6148
   macro avg       0.88      0.82      0.85      6148
weighted avg       0.92      0.93      0.92      6148



In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.25, random_state=10)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.82      0.70      0.75       961
           1       0.95      0.97      0.96      5187

    accuracy                           0.93      6148
   macro avg       0.88      0.83      0.85      6148
weighted avg       0.92      0.93      0.93      6148



In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.25, random_state=42)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.3, random_state=0)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.86      0.64      0.73       795
           1       0.94      0.98      0.96      4328

    accuracy                           0.93      5123
   macro avg       0.90      0.81      0.84      5123
weighted avg       0.92      0.93      0.92      5123



In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.3, random_state=10)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


===============================pemodelan SVM===============================
=========================CLASSIFICATION REPORT SVM=========================
              precision    recall  f1-score   support

           0       0.86      0.67      0.75       787
           1       0.94      0.98      0.96      4336

    accuracy                           0.93      5123
   macro avg       0.90      0.82      0.86      5123
weighted avg       0.93      0.93      0.93      5123



In [25]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
from scipy.sparse import csr_matrix


# Split the data into training and testing sets
sparse_tfidf_matrix = csr_matrix(dense_tfidf_matrix)  # Might be unnecessary if dense_tfidf_matrix is desired format
dense_tfidf_matrix = sparse_tfidf_matrix.toarray()  # Convert sparse matrix to dense array
X_train, X_test, y_train, y_test = train_test_split(dense_tfidf_matrix, Sentiment, test_size=0.3, random_state=42)

print("PEMODELAN DENGAN NAIVE BAYES".center(75, "="))
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_nb = round(accuracy_score(y_test, y_pred) * 100, 2)
acc_gaussian = round(model.score(X_train, y_train) * 100, 2)

# Perhitungan confusion matrix
print('CLASSIFICATION REPORT NAIVE BAYES'.center(75, '='))
print(classification_report(y_test, y_pred))


MemoryError: Unable to allocate 3.49 GiB for an array with shape (20491, 22863) and data type float64